
# Reading a Radio Light Curve: Peak Times and Slopes

**Objective:** fit a simple analytic function to multi-band radio light curves, read off the peak time and the
rise and decay slopes in each band, and check that the fitted frequency dependence of the peak time matches the
true peak times of the source.

A phenomenological fit makes no physical assumption, so its parameters are directly comparable between
sources. Here the "observations" come from a physical model of a supernova in a steady wind (the SN 1993J-like
model of `the previous example <sphx_glr_auto_examples_modeling_plot_sn1993j_radio_sed.py>`), observed at
three frequencies with 10% errors. Because the true source is known, the fitted quantities can be compared with
what the physics predicts.

We use :class:`~trilobite.models.generic.light_curve.SmoothedBrokenPowerLawTime`, a rising power law
$F \propto t^{\alpha_1}$ that turns smoothly into a falling power law $F \propto t^{\alpha_2}$ around a
break time $t_b$. The smoothing parameter $s$ sets the width of the peak, and is negative for a peak.
Radio supernovae fade as power laws, not exponentials, so this is a better choice than a FRED profile.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from scipy.optimize import curve_fit

from trilobite.dynamics.shocks.core import StrongColdShockConditions
from trilobite.models.generic.light_curve import SmoothedBrokenPowerLawTime
from trilobite.radiation.synchrotron.cooling import SynchrotronRadiativeCoolingEngine
from trilobite.radiation.synchrotron.electron_distributions import PowerLaw
from trilobite.radiation.synchrotron.SEDs.numerical import NonRelativisticSphericalSynchrotronEngine
from trilobite.utils.plot_utils import set_plot_style

set_plot_style()
rng = np.random.default_rng(4)

## The Source

The source is SN 1993J with the parameters of the previous example (Fransson & Bjornsson 1998): a supernova at
3.63 Mpc in a wind of $5\times10^{-5}\,M_\odot\,{\rm yr^{-1}}$ at 10 km/s, a shock at
$2.2\times10^4$ km/s that decelerates after 100 days, and the same magnetic and electron energy fractions
and effective emitting area ($f_A = 0.1$).
The function below turns the shock, the magnetic field and the numerical synchrotron engine into a flux density
at a given frequency and set of epochs.



In [ ]:
D, M_dot, v_wind = 3.63 * u.Mpc, 5.0e-5 * u.Msun / u.yr, 10.0 * u.km / u.s
V_0, t_p, m = 22_000.0 * u.km / u.s, 100.0 * u.day, 0.74
p, gamma_min, gamma_max = 2.1, 1.0, 1.0e6
epsilon_B, epsilon_E = 0.14 * 8 / 9, 5.1e-4 * 8 / 9

engine = NonRelativisticSphericalSynchrotronEngine()
engine.load_avg_first_kernel()
cooling = SynchrotronRadiativeCoolingEngine()
gamma = np.geomspace(gamma_min, gamma_max, 500)


def physical_light_curve(t, freq):
    """Flux density of the supernova at frequency ``freq`` and epochs ``t``."""
    x = (t / t_p).decompose().value
    V = V_0 * np.where(x < 1, 1.0, x ** (m - 1))
    R = np.where(x < 1, V_0 * t, V_0 * t_p * (1 + (x**m - 1) / m)).to(u.cm)
    rho = (M_dot / (4 * np.pi * R**2 * v_wind)).to(u.g / u.cm**3)
    e_th = StrongColdShockConditions.compute_post_shock_thermal_energy_density(shock_velocity=V, upstream_density=rho)
    B = np.sqrt(8 * np.pi * epsilon_B * e_th.to_value(u.erg / u.cm**3)) * u.G
    gamma_c = cooling.compute_cooling_gamma(B=B, t=t)

    flux = np.empty(t.size) * u.mJy
    for i in range(t.size):
        N0 = PowerLaw.normalize_from_magnetic_field(
            B[i], epsilon_B, epsilon_E, p=p, gamma_min=gamma_min, gamma_max=gamma_max
        ).to_value(u.cm**-3)
        N = N0 * np.where(gamma < gamma_c[i], gamma**-p, gamma_c[i] * gamma ** (-p - 1))
        flux[i] = engine.compute_flux_density(
            freq, R=R[i], B=B[i], N=N, gamma=gamma, f_V=0.5, f_A=0.1, luminosity_distance=D
        ).to(u.mJy)
    return flux

## Simulated Monitoring Campaign

We observe at 1.4, 5 and 22 GHz on fourteen epochs between 12 and 1800 days, with 10% Gaussian errors, which is
typical for calibration-limited radio monitoring.



In [ ]:
freqs = [1.4, 5.0, 22.0] * u.GHz
t_obs = np.array([12, 18, 26, 38, 55, 80, 115, 165, 240, 350, 520, 800, 1200, 1800]) * u.day
t_fine = np.geomspace(10.0, 2000.0, 60) * u.day

data = {}
for freq in freqs:
    truth = physical_light_curve(t_obs, freq)
    err = 0.1 * truth
    data[freq.value] = (truth + rng.normal(scale=err.value) * u.mJy, err, physical_light_curve(t_fine, freq))

## Fitting Each Band

We fit :class:`~trilobite.models.generic.light_curve.SmoothedBrokenPowerLawTime` to each band in log space, weighting
by the errors. The model has the parameters ``A`` (flux at the break), ``t_b`` (break time), ``alpha_1`` (rise
slope), ``alpha_2`` (decay slope) and ``s`` (smoothing, negative for a peak), all of which are fitted.



In [ ]:
lc_model = SmoothedBrokenPowerLawTime()


def log_flux(t, log_A, t_b, alpha_1, alpha_2, s):
    """Natural log of the smoothed broken power-law flux, as a function of the fit parameters."""
    params = {"A": 10**log_A, "t_b": t_b, "alpha_1": alpha_1, "alpha_2": alpha_2, "s": s, "C": 0.0}
    return np.log(lc_model.forward_model({"t": t}, params).flux)


fits = {}
for freq in freqs:
    flux, err, _ = data[freq.value]
    k = np.argmax(flux)
    guess = [np.log10(flux[k].value), t_obs[k].value, 2.0, -1.0, -0.3]
    popt, pcov = curve_fit(
        log_flux,
        t_obs.value,
        np.log(flux.value),
        p0=guess,
        sigma=(err / flux).value,
        absolute_sigma=True,
        bounds=([-3, 5, 0.1, -5, -3], [5, 2000, 6, 0, -0.05]),
    )
    fits[freq.value] = (popt, np.sqrt(np.diag(pcov)))

fig, (ax_lc, ax_tb) = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)

for (freq, (flux, err, _)), color in zip(data.items(), ["C3", "C1", "C0"]):
    popt, _ = fits[freq]
    ax_lc.errorbar(t_obs.value, flux.value, err.value, fmt="o", color=color, capsize=3, label=f"{freq:g} GHz")
    ax_lc.loglog(t_fine.value, np.exp(log_flux(t_fine.value, *popt)), color=color, lw=1.8)
ax_lc.set(xlabel="Time since explosion [day]", ylabel="Flux density [mJy]", title="Smoothed broken power-law fits")
ax_lc.legend()

## Peak Time Versus Frequency

The peak of each band occurs when the self-absorption frequency passes through that band, so the peak time falls
with frequency. For a smooth break the parameter $t_b$ is only where the two asymptotes cross, so we
read the peak time from the maximum of each fitted curve and compare it with the true peak of the source. A
power law through the fitted peak times gives the frequency dependence.



In [ ]:
nu_GHz = freqs.value
t_peak_fit = np.array([t_fine.value[np.argmax(log_flux(t_fine.value, *fits[f][0]))] for f in nu_GHz])
t_peak_true = np.array([t_fine.value[np.argmax(data[f][2].value)] for f in nu_GHz])
slope, intercept = np.polyfit(np.log(nu_GHz), np.log(t_peak_fit), 1)
slope_true = np.polyfit(np.log(nu_GHz), np.log(t_peak_true), 1)[0]

ax_tb.loglog(nu_GHz, t_peak_true, "s", ms=9, mfc="none", color="k", label="True peak time")
ax_tb.loglog(nu_GHz, t_peak_fit, "o", ms=6, label="Fitted peak time")
ax_tb.loglog(nu_GHz, np.exp(intercept) * nu_GHz**slope, "k--", label=rf"$t_{{\rm pk}} \propto \nu^{{{slope:.2f}}}$")
ax_tb.set(xlabel="Frequency [GHz]", ylabel="Peak time [day]", title="Peak time versus frequency")
ax_tb.legend()
plt.show()

for freq, tf, tt, (_, _, a1, a2, s_fit) in zip(nu_GHz, t_peak_fit, t_peak_true, [fits[f][0] for f in nu_GHz]):
    print(f"{freq:>5g} GHz: peak {tf:4.0f} d (true {tt:4.0f} d), rise t^{a1:.2f}, decay t^{a2:.2f}, s = {s_fit:.2f}")
print(f"Frequency dependence of the peak time: fitted nu^{slope:.2f}, true nu^{slope_true:.2f}")

**Outcome:** the fitted peak times fall as $\nu^{-0.78}$, against $\nu^{-0.75}$ for the true peak
times of the model, and each fitted peak is within about 10% of the true peak. The three bands share a rise of
about $t^{2.5}$ and a decay of $t^{-1}$ to $t^{-1.4}$. The fitted smoothing $s$ differs
between bands, which shows that the width of the peak is much less well constrained than its position.

